In [ ]:
# AI-BASED REAL-TIME ANOMALY DETECTION & AUTONOMOUS DECISION SUPPORT        ║

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
import seaborn as sns
from IPython.display import display, HTML, Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
    mean_absolute_error, mean_squared_error
)
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
import tensorflow as tf

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

# ── Global palette ──────────────────────────────────────────────────────────────
C = {
    'bg':        '#06090f',   # near-black
    'panel':     '#0b1120',   # card background
    'border':    '#162b45',   # panel borders
    'teal':      '#00e5ff',   # primary accent
    'amber':     '#ffaa00',   # warning
    'crimson':   '#ff1f4b',   # critical
    'green':     '#00ff9d',   # nominal
    'muted':     '#3d6a8a',   # labels / ticks
    'text':      '#c5dff0',   # body text
    'purple':    '#a78bfa',   # secondary accent
}

plt.rcParams.update({
    'figure.facecolor':  C['bg'],
    'axes.facecolor':    C['panel'],
    'axes.edgecolor':    C['border'],
    'axes.labelcolor':   C['muted'],
    'xtick.color':       C['muted'],
    'ytick.color':       C['muted'],
    'text.color':        C['text'],
    'grid.color':        C['border'],
    'grid.alpha':        0.5,
    'font.family':       'monospace',
    'axes.titlepad':     8,
})

# ── Banner ──────────────────────────────────────────────────────────────────────
display(HTML(f"""
<div style="
  background: linear-gradient(160deg, #06090f 0%, #0b1a30 60%, #061020 100%);
  border: 1px solid #162b45;
  border-top: 3px solid #00e5ff;
  border-radius: 10px;
  padding: 22px 32px;
  margin-bottom: 18px;
  font-family: 'Courier New', monospace;
  position: relative;
  overflow: hidden;
">
  <div style="
    position: absolute; top: 0; right: 0; width: 180px; height: 100%;
    background: radial-gradient(ellipse at top right, rgba(0,229,255,0.06), transparent 70%);
  "></div>

  <div style="color:#00e5ff; font-size:10px; letter-spacing:4px; margin-bottom:6px; opacity:0.7">
    ◈ INDIA SPACE ACADEMY — MISSION CONTROL SYSTEM
  </div>

  <div style="color:#c5dff0; font-size:20px; font-weight:bold; letter-spacing:2px; margin-bottom:4px">
    DEEP SPACE ENGINE HEALTH MONITOR
  </div>

  <div style="color:#3d6a8a; font-size:10px; letter-spacing:3px; margin-bottom:14px">
    NASA CMAPSS FD001 · ISOLATION FOREST · AUTOENCODER · LSTM · RUL PREDICTION
  </div>

  <div style="display:flex; gap:24px; font-size:10px; letter-spacing:2px;">
    <span style="color:#00ff9d">● SYSTEM ONLINE</span>
    <span style="color:#ffaa00">◆ MODELS: 4</span>
    <span style="color:#00e5ff">▶ ENGINES: 100</span>
    <span style="color:#a78bfa">⬡ VERSION 2.0</span>
  </div>
</div>
"""))

def section_header(n, total, label):
    display(HTML(f"""
    <div style="
      display:flex; align-items:center; gap:12px;
      font-family:'Courier New',monospace; margin: 14px 0 6px 0;
    ">
      <span style="color:#00e5ff; font-size:11px; opacity:0.6">[{n}/{total}]</span>
      <span style="color:#00e5ff; font-size:12px; letter-spacing:3px; font-weight:bold">{label}</span>
      <span style="flex:1; height:1px; background:linear-gradient(90deg,#162b45,transparent)"></span>
    </div>
    """))


# 1 ·  DATA LOADING & FEATURE ENGINEERING
section_header(1, 5, 'LOADING TELEMETRY DATA')

PATH = '/kaggle/input/datasets/bishals098/nasa-turbofan-engine-degradation-simulation'
cols = ['unit', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]

train    = pd.read_csv(f'{PATH}/train_FD001.txt', sep=r'\s+', header=None, names=cols)
test     = pd.read_csv(f'{PATH}/test_FD001.txt',  sep=r'\s+', header=None, names=cols)
rul_true = pd.read_csv(f'{PATH}/RUL_FD001.txt',   header=None, names=['rul'])

# RUL target
train['max_cycle'] = train.groupby('unit')['cycle'].transform('max')
train['RUL']       = train['max_cycle'] - train['cycle']
train.drop(columns=['max_cycle'], inplace=True)

# Drop near-constant and non-sensor columns
DROP  = ['unit', 'cycle', 'os1', 'os2', 'os3', 's1', 's5', 's6', 's10', 's16', 's18', 's19']
SCOLS = [c for c in cols if c not in DROP]
N     = len(SCOLS)

DANGER  = 30      # cycles — critical threshold
WINDOW  = 30      # LSTM look-back
MAX_RUL = 125     # cap for normalisation

scaler   = MinMaxScaler()
X_train  = scaler.fit_transform(train[SCOLS])
X_test   = scaler.transform(test[SCOLS])

labels   = (train['RUL'] < DANGER).astype(int).values   # binary anomaly ground truth
X_health = X_train[labels == 0]                          # healthy-only data for unsupervised models
units    = train['unit'].values

# ── Summary ─────────────────────────────────────────────────────────────────────
print(f'  ┌─────────────────────────────────────────┐')
print(f'  │  Train rows   : {len(train):>8,}                │')
print(f'  │  Test rows    : {len(test):>8,}                │')
print(f'  │  Features     : {N:>8}                │')
print(f'  │  Normal pts   : {(labels==0).sum():>8,}                │')
print(f'  │  Degraded pts : {(labels==1).sum():>8,}                │')
print(f'  └─────────────────────────────────────────┘')


# 2 ·  MODEL TRAINING
section_header(2, 5, 'TRAINING MODELS')

# ── 2a: Isolation Forest ────────────────────────────────────────────────────────
print('  ▶ Isolation Forest ...')
iso       = IsolationForest(n_estimators=150, contamination=0.05,
                            max_features=0.8, random_state=42)
iso.fit(X_health)
iso_preds = (iso.predict(X_train) == -1).astype(int)
iso_score = iso.decision_function(X_train)              # anomaly score (lower = more anomalous)
print('    ✓ done')

# ── 2b: Autoencoder ─────────────────────────────────────────────────────────────
print('  ▶ Autoencoder ...')

def build_autoencoder(n_features):
    inp = keras.Input(shape=(n_features,))
    # encoder
    x = layers.Dense(16, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(8,  activation='relu')(x)
    x = layers.Dense(4,  activation='relu')(x)   # bottleneck
    # decoder
    x = layers.Dense(8,  activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(n_features, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=optimizers.Adam(1e-3), loss='mse')
    return model

ae = build_autoencoder(N)
ae.fit(X_health, X_health, epochs=40, batch_size=64, validation_split=0.1, verbose=0,
       callbacks=[
           callbacks.EarlyStopping(patience=6, restore_best_weights=True),
           callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-6)
       ])

ae_err_health = np.mean((X_health - ae.predict(X_health, verbose=0))**2, axis=1)
ae_thr        = np.percentile(ae_err_health, 95)
ae_err        = np.mean((X_train - ae.predict(X_train, verbose=0))**2, axis=1)
ae_preds      = (ae_err > ae_thr).astype(int)
print('    ✓ done')

# ── 2c: LSTM Anomaly Classifier ─────────────────────────────────────────────────
print('  ▶ LSTM Anomaly Classifier ...')

def make_windows(X, y, units_arr, w):
    Xl, yl, idx = [], [], []
    for uid in np.unique(units_arr):
        m  = np.where(units_arr == uid)[0]
        xs, ys = X[m], y[m]
        for i in range(len(xs) - w):
            Xl.append(xs[i:i+w]);  yl.append(ys[i+w]);  idx.append(m[i+w])
    return np.array(Xl), np.array(yl), np.array(idx)

Xw, yw, win_idx = make_windows(X_train, labels, units, WINDOW)

# Handle class imbalance
neg, pos   = (yw == 0).sum(), (yw == 1).sum()
cw         = {0: 1.0, 1: neg / pos}

lstm_ad = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(WINDOW, N)),
    layers.Dropout(0.25),
    layers.LSTM(32),
    layers.Dropout(0.25),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid')
])
lstm_ad.compile(optimizer=optimizers.Adam(1e-3),
                loss='binary_crossentropy', metrics=['accuracy'])
lstm_ad.fit(Xw, yw, epochs=25, batch_size=256,
            validation_split=0.1, class_weight=cw, verbose=0,
            callbacks=[callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

raw_probs  = lstm_ad.predict(Xw, verbose=0).flatten()
lstm_probs = np.zeros(len(train))
lstm_probs[win_idx] = raw_probs
lstm_preds = np.zeros(len(train), dtype=int)
lstm_preds[win_idx] = (raw_probs >= 0.5).astype(int)
print('    ✓ done')

# ── 2d: LSTM RUL Regressor ──────────────────────────────────────────────────────
print('  ▶ LSTM RUL Regressor ...')
train['RUL_cap'] = train['RUL'].clip(upper=MAX_RUL)

def make_windows_rul(X, rul, units_arr, w):
    Xl, yl = [], []
    for uid in np.unique(units_arr):
        m  = units_arr == uid
        xs, rs = X[m], rul[m]
        for i in range(len(xs) - w):
            Xl.append(xs[i:i+w]);  yl.append(rs[i+w])
    return np.array(Xl), np.array(yl)

Xr, yr = make_windows_rul(X_train, train['RUL_cap'].values, units, WINDOW)

# Cosine decay LR schedule
total_steps = (len(Xr) // 256) * 35
lr_sched    = optimizers.schedules.CosineDecay(1e-3, decay_steps=total_steps, alpha=1e-5)

lstm_rul = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(WINDOW, N)),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid')
])
lstm_rul.compile(optimizer=optimizers.Adam(lr_sched), loss='mse', metrics=['mae'])
lstm_rul.fit(Xr, yr / MAX_RUL, epochs=35, batch_size=256,
             validation_split=0.1, verbose=0,
             callbacks=[callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
print('    ✓ done')

# ── Test-set RUL ─────────────────────────────────────────────────────────────────
def get_last_window(X, units_arr, w):
    Xl = []
    for uid in sorted(np.unique(units_arr)):
        m  = units_arr == uid;  xs = X[m]
        if len(xs) < w:
            xs = np.vstack([np.tile(xs[0], (w - len(xs), 1)), xs])
        Xl.append(xs[-w:])
    return np.array(Xl)

X_test_win = get_last_window(X_test, test['unit'].values, WINDOW)
rul_pred   = lstm_rul.predict(X_test_win, verbose=0).flatten() * MAX_RUL
rul_actual = rul_true['rul'].values
mae  = mean_absolute_error(rul_actual, rul_pred)
rmse = np.sqrt(mean_squared_error(rul_actual, rul_pred))


# 3 ·  RESULTS & VISUALISATIONS
section_header(3, 5, 'RESULTS & VISUALISATIONS')

# ── Performance table ───────────────────────────────────────────────────────────
print(f'\n  {"MODEL":<22} {"PREC":>7} {"REC":>7} {"F1":>7} {"AUC":>7}')
print(f'  {"─"*52}')
results = {}
for name, preds, proba in [
    ('Isolation Forest', iso_preds,  -iso_score),
    ('Autoencoder',      ae_preds,    ae_err),
    ('LSTM Classifier',  lstm_preds,  lstm_probs),
]:
    p = precision_score(labels, preds, zero_division=0)
    r = recall_score(labels, preds, zero_division=0)
    f = f1_score(labels, preds, zero_division=0)
    try:
        a = roc_auc_score(labels, proba)
    except Exception:
        a = float('nan')
    results[name] = dict(precision=p, recall=r, f1=f, auc=a, preds=preds)
    print(f'  {name:<22} {p:>7.3f} {r:>7.3f} {f:>7.3f} {a:>7.3f}')

c_e = (rul_pred < 30).sum()
w_e = ((rul_pred >= 30) & (rul_pred < 80)).sum()
n_e = (rul_pred >= 80).sum()
print(f'\n  RUL  →  MAE={mae:.2f}  RMSE={rmse:.2f}')
print(f'  Fleet  →  ✅ normal={n_e}   warning={w_e}   critical={c_e}')

# ── Fig 1: Confusion matrices + model comparison bar ────────────────────────────
fig = plt.figure(figsize=(16, 8))
fig.suptitle('ANOMALY DETECTION — MODEL EVALUATION', color=C['teal'],
             fontsize=13, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.45)

model_list = [('Isolation Forest', results['Isolation Forest']['preds']),
              ('Autoencoder',      results['Autoencoder']['preds']),
              ('LSTM Classifier',  results['LSTM Classifier']['preds'])]

for col_i, (name, preds) in enumerate(model_list):
    ax = fig.add_subplot(gs[0, col_i])
    cm = confusion_matrix(labels, preds)
    # custom heatmap colours
    cmap = mcolors.LinearSegmentedColormap.from_list(
        'space', [C['panel'], C['teal']], N=256)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Normal', 'Degraded'],
                yticklabels=['Normal', 'Degraded'],
                linewidths=1, linecolor=C['bg'],
                annot_kws={'fontsize': 11, 'color': C['text']},
                cbar=False)
    ax.set_title(name, color=C['teal'], fontsize=9)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('Actual',    fontsize=8)
    ax.tick_params(labelsize=7)

# Model metric comparison bar chart
ax_bar = fig.add_subplot(gs[0, 3])
metrics = ['precision', 'recall', 'f1', 'auc']
x       = np.arange(len(metrics))
w       = 0.22
pal     = [C['teal'], C['purple'], C['amber']]
for i, (name, _) in enumerate(model_list):
    vals = [results[name][m] for m in metrics]
    bars = ax_bar.bar(x + i * w - w, vals, w, label=name.split()[0],
                      color=pal[i], alpha=0.85, edgecolor=C['bg'])
ax_bar.set_xticks(x - w / 2)
ax_bar.set_xticklabels(['Prec', 'Rec', 'F1', 'AUC'], fontsize=8)
ax_bar.set_ylim(0, 1.12)
ax_bar.set_title('Metric Comparison', color=C['teal'], fontsize=9)
ax_bar.legend(fontsize=7, facecolor=C['panel'], edgecolor=C['border'],
              labelcolor=C['text'], loc='upper right')
ax_bar.grid(True, axis='y', alpha=0.3)
ax_bar.set_facecolor(C['panel'])

# ── RUL scatter + error hist ─────────────────────────────────────────────────────
ax_sc  = fig.add_subplot(gs[1, :2])
ax_his = fig.add_subplot(gs[1, 2:])

status_color = np.where(rul_pred < 30, C['crimson'],
               np.where(rul_pred < 80, C['amber'], C['green']))
ax_sc.scatter(rul_actual, rul_pred, c=status_color, alpha=0.55, s=25, zorder=3)
mv = max(rul_actual.max(), rul_pred.max())
ax_sc.plot([0, mv], [0, mv], color='#ffffff', ls='--', lw=1.2, label='perfect', alpha=0.4)
ax_sc.fill_between([0, mv], [0, mv * 0.8], [0, mv * 1.2], color=C['teal'], alpha=0.06,
                   label='±20 % band')
ax_sc.set(title=f'Predicted vs Actual RUL  (MAE={mae:.1f} | RMSE={rmse:.1f})',
          xlabel='True RUL (cycles)', ylabel='Predicted RUL (cycles)')
ax_sc.legend(fontsize=7, facecolor=C['panel'], edgecolor=C['border'], labelcolor=C['text'])
ax_sc.grid(True, alpha=0.3)
legend_elems = [Line2D([0], [0], marker='o', color='w', markerfacecolor=C['green'],   markersize=7, label='Normal (≥80)'),
                Line2D([0], [0], marker='o', color='w', markerfacecolor=C['amber'],   markersize=7, label='Warning (30–80)'),
                Line2D([0], [0], marker='o', color='w', markerfacecolor=C['crimson'], markersize=7, label='Critical (<30)')]
ax_sc.legend(handles=legend_elems, fontsize=7, facecolor=C['panel'],
             edgecolor=C['border'], labelcolor=C['text'], loc='upper left')

errors = rul_pred - rul_actual
ax_his.hist(errors[errors < 0],  bins=20, color=C['crimson'], alpha=0.75,
            label='over-estimated risk', edgecolor=C['bg'])
ax_his.hist(errors[errors >= 0], bins=20, color=C['green'],   alpha=0.75,
            label='under-estimated risk', edgecolor=C['bg'])
ax_his.axvline(0,           color='#ffffff', ls='--', lw=1.2, alpha=0.5)
ax_his.axvline(errors.mean(), color=C['amber'],  ls=':', lw=1.5,
               label=f'mean error {errors.mean():.1f}')
ax_his.set(title='Prediction Error Distribution', xlabel='Error (cycles)', ylabel='Count')
ax_his.legend(fontsize=7, facecolor=C['panel'], edgecolor=C['border'], labelcolor=C['text'])
ax_his.grid(True, alpha=0.3)

for ax in [ax_sc, ax_his]:
    ax.set_facecolor(C['panel'])

plt.tight_layout()
plt.savefig('plot_model_evaluation.png', dpi=130, bbox_inches='tight',
            facecolor=C['bg'])
plt.show()
print('  ✓ plot_model_evaluation.png saved')

# ── Fig 2: Fleet lollipop chart ──────────────────────────────────────────────────
order     = np.argsort(rul_pred)
sorted_rul = rul_pred[order]
eng_ids    = np.arange(1, 101)[order]

fig, ax = plt.subplots(figsize=(15, 5))
fig.suptitle('FLEET HEALTH OVERVIEW — 100 TEST ENGINES  (sorted by RUL)',
             color=C['teal'], fontsize=12, fontweight='bold')

bar_colors = [C['crimson'] if r < 30 else C['amber'] if r < 80 else C['green']
              for r in sorted_rul]

for i, (rul_v, col) in enumerate(zip(sorted_rul, bar_colors)):
    ax.vlines(i, 0, rul_v, color=col, lw=1.4, alpha=0.55)
    ax.scatter(i, rul_v, color=col, s=18, zorder=4, alpha=0.9)

ax.axhline(30, color=C['crimson'], ls='--', lw=1.2, alpha=0.7, label='Critical < 30 cycles')
ax.axhline(80, color=C['amber'],   ls='--', lw=1.2, alpha=0.7, label='Warning  < 80 cycles')
ax.fill_between(range(100), 0, 30, color=C['crimson'], alpha=0.05)
ax.fill_between(range(100), 30, 80, color=C['amber'],  alpha=0.04)

ax.set_xticks(range(100))
ax.set_xticklabels([str(e) for e in eng_ids], fontsize=5.5, rotation=90)
ax.set_ylabel('Predicted RUL (cycles)', fontsize=9)
ax.set_xlabel('Engine ID (sorted by RUL)', fontsize=9)
ax.legend(fontsize=8, facecolor=C['panel'], edgecolor=C['border'], labelcolor=C['text'])
ax.grid(True, axis='y', alpha=0.3)

# Annotation boxes
for label_txt, rul_v, col, ypos in [
    (f'CRITICAL\n{c_e} engines', 14,  C['crimson'], 0.85),
    (f'WARNING\n{w_e} engines',  55,  C['amber'],   0.85),
    (f'NORMAL\n{n_e} engines',   100, C['green'],   0.85),
]:
    ax.text(0.12 if 'CRIT' in label_txt else 0.45 if 'WARN' in label_txt else 0.80,
            ypos, label_txt, transform=ax.transAxes,
            ha='center', va='center', color=col, fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=C['panel'],
                      edgecolor=col, alpha=0.9))

plt.tight_layout()
plt.savefig('plot_fleet.png', dpi=130, bbox_inches='tight', facecolor=C['bg'])
plt.show()
print('  ✓ plot_fleet.png saved')


# 4 ·  AUTONOMOUS DECISION ENGINE
section_header(4, 5, 'AUTONOMOUS DECISION ENGINE')

def risk_score(iso_flag, ae_flag, lstm_flag, pred_rul):
    """Weighted composite risk: model agreement + RUL urgency."""
    model_vote = (0.25 * iso_flag + 0.30 * ae_flag + 0.45 * lstm_flag)
    rul_urgency = 1.0 - min(pred_rul / MAX_RUL, 1.0)
    return round(0.35 * model_vote + 0.65 * rul_urgency, 4)

ACTION_MAP = [
    (0.20, 'NOMINAL',     'Continue normal operations',      C['green']),
    (0.40, 'MONITOR',     'Increase telemetry sampling',     C['teal']),
    (0.60, 'WARNING',     'Reduce engine load by 20 %',      C['amber']),
    (0.75, 'HIGH RISK',   'Activate redundant systems',      '#ff6600'),
    (1.01, 'EMERGENCY',   'Initiate controlled shutdown',    C['crimson']),
]

def get_action(risk):
    for thr, status, action, color in ACTION_MAP:
        if risk < thr:
            return status, action, color
    return ACTION_MAP[-1][1], ACTION_MAP[-1][2], ACTION_MAP[-1][3]

action_counts = {s: 0 for _, s, _, _ in ACTION_MAP}
decisions     = []
for i in range(len(rul_pred)):
    rows = np.where(units == (i + 1))[0]
    iv   = int(iso_preds[rows[-1]])   if len(rows) else 0
    av   = int(ae_preds[rows[-1]])    if len(rows) else 0
    lv   = int(lstm_preds[rows[-1]])  if len(rows) else 0
    r    = risk_score(iv, av, lv, rul_pred[i])
    st, ac, col = get_action(r)
    action_counts[st] += 1
    decisions.append(dict(engine=i+1, rul=round(rul_pred[i],1), risk=r,
                          status=st, action=ac))

print(f'\n  {"ENG":>4} {"RUL":>7} {"RISK":>7}  {"STATUS":<12} ACTION')
print(f'  {"─"*62}')
# Print first 15 as sample
for d in sorted(decisions, key=lambda x: x['rul'])[:15]:
    print(f'  {d["engine"]:>4} {d["rul"]:>7.1f} {d["risk"]:>7.4f}  {d["status"]:<12} {d["action"]}')
print(f'  ... (showing 15 of 100)')
print(f'\n  Action distribution: {action_counts}')


# 5 ·  REAL-TIME ANIMATION  (enhanced 4-panel layout)
section_header(5, 5, 'REAL-TIME MONITOR — RENDERING ANIMATION')

WATCH_UNIT   = 3
WATCH_SENSOR = 's2'
TRAIL        = 60

eng_mask  = units == WATCH_UNIT
eng_data  = train[eng_mask].reset_index(drop=True)
eng_X     = X_train[eng_mask]
eng_rul   = train['RUL'].values[eng_mask]
eng_iso   = iso_preds[eng_mask]
eng_ae    = ae_preds[eng_mask]
eng_lstm  = lstm_preds[eng_mask]
total_cyc = len(eng_data)
danger_cyc= eng_data['cycle'].max() - DANGER

# precompute per-cycle risk using all three model flags
risks = np.array([
    risk_score(int(eng_iso[i]), int(eng_ae[i]), int(eng_lstm[i]), eng_rul[i])
    for i in range(total_cyc)
])

def status_color(rul_v, risk_v):
    if rul_v < DANGER or risk_v >= 0.60: return C['crimson']
    if rul_v < 80    or risk_v >= 0.40:  return C['amber']
    return C['green']

alerts = []

fig = plt.figure(figsize=(15, 10), facecolor=C['bg'])
fig.suptitle(
    f'REAL-TIME ENGINE {WATCH_UNIT:02d} MONITOR  ·  SENSOR {WATCH_SENSOR.upper()}',
    color=C['teal'], fontsize=13, fontweight='bold', y=0.99
)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.60, wspace=0.38,
                       left=0.07, right=0.97, top=0.94, bottom=0.06)

ax_s  = fig.add_subplot(gs[0, :])      # sensor stream (full width)
ax_r  = fig.add_subplot(gs[1, :2])     # risk timeline
ax_g  = fig.add_subplot(gs[1, 2])      # RUL gauge
ax_h  = fig.add_subplot(gs[2, :2])     # model-agreement heatmap strip
ax_a  = fig.add_subplot(gs[2, 2])      # alert log

for ax in [ax_s, ax_r, ax_g, ax_h, ax_a]:
    ax.set_facecolor(C['panel'])
    for sp in ax.spines.values():
        sp.set_edgecolor(C['border'])

def animate(frame):
    i       = min(frame + 1, total_cyc - 1)
    start   = max(0, i - TRAIL)
    cyc_win = eng_data['cycle'].values[start:i+1]
    sns_win = eng_data[WATCH_SENSOR].values[start:i+1]
    iso_win = eng_iso[start:i+1]
    ae_win  = eng_ae[start:i+1]
    lstm_win= eng_lstm[start:i+1]
    any_anom= (iso_win == 1) | (ae_win == 1) | (lstm_win == 1)
    cur_rul = eng_rul[i]
    cur_risk= risks[i]
    cur_cyc = eng_data['cycle'].values[i]
    col     = status_color(cur_rul, cur_risk)
    st, ac, _ = get_action(cur_risk)

    # ── Panel 1: Sensor stream ─────────────────────────────────────────────────
    ax_s.cla(); ax_s.set_facecolor(C['panel'])
    for sp in ax_s.spines.values(): sp.set_edgecolor(C['border'])

    # gradient fill from current status colour to transparent
    ax_s.fill_between(cyc_win, sns_win, alpha=0.12, color=col)
    ax_s.plot(cyc_win, sns_win, color=col, lw=1.6, zorder=3)

    # highlight anomaly points
    if any_anom.any():
        ax_s.scatter(cyc_win[any_anom], sns_win[any_anom],
                     color=C['crimson'], s=35, zorder=6,
                     edgecolors='white', linewidths=0.4,
                     label='⚠ anomaly detected')

    # danger zone shading
    ax_s.axvspan(danger_cyc, cyc_win[-1] + 5, alpha=0.06, color=C['crimson'])
    ax_s.axvline(danger_cyc, color=C['crimson'], ls=':', lw=1, alpha=0.5,
                 label='danger threshold')

    ax_s.set_xlim(cyc_win[0], cyc_win[0] + TRAIL + 2)
    ax_s.set_ylabel(WATCH_SENSOR.upper(), fontsize=8)
    ax_s.set_title(
        f'CYCLE {cur_cyc:3d}   RUL {cur_rul:4.0f} cyc   '
        f'RISK {cur_risk:.3f}   [{st}]',
        color=col, fontsize=9, fontweight='bold', pad=4)
    ax_s.legend(fontsize=7, loc='upper left',
                facecolor=C['panel'], edgecolor=C['border'],
                labelcolor=C['text'])
    ax_s.grid(True, alpha=0.25)

    # ── Panel 2: Risk timeline ─────────────────────────────────────────────────
    ax_r.cla(); ax_r.set_facecolor(C['panel'])
    for sp in ax_r.spines.values(): sp.set_edgecolor(C['border'])

    risk_win = risks[start:i+1]
    for j in range(len(cyc_win) - 1):
        rc = status_color(eng_rul[start+j], risk_win[j])
        ax_r.plot(cyc_win[j:j+2], risk_win[j:j+2], color=rc, lw=1.8)

    for thr, lbl, tc in [(0.60, 'emergency', C['crimson']),
                          (0.40, 'warning',   C['amber']),
                          (0.20, 'monitor',   C['teal'])]:
        ax_r.axhline(thr, color=tc, ls='--', lw=0.9, alpha=0.6, label=lbl)

    ax_r.set_ylim(0, 1)
    ax_r.set_xlim(cyc_win[0], cyc_win[0] + TRAIL + 2)
    ax_r.set_ylabel('Composite Risk', fontsize=8)
    ax_r.set_xlabel('Cycle',          fontsize=8)
    ax_r.set_title('COMPOSITE RISK SCORE', color=C['teal'], fontsize=9, fontweight='bold')
    ax_r.legend(fontsize=7, loc='upper left',
                facecolor=C['panel'], edgecolor=C['border'], labelcolor=C['text'])
    ax_r.grid(True, alpha=0.25)

    # ── Panel 3: RUL gauge ────────────────────────────────────────────────────
    ax_g.cla(); ax_g.axis('off')
    ax_g.set_facecolor(C['panel'])
    ax_g.set_xlim(0, 1); ax_g.set_ylim(0, 1)
    pct = min(cur_rul / MAX_RUL, 1.0)

    # Track bar background
    ax_g.add_patch(FancyBboxPatch((0.10, 0.20), 0.80, 0.09,
        boxstyle='round,pad=0.01', facecolor=C['border'], edgecolor='none'))
    # Fill
    ax_g.add_patch(FancyBboxPatch((0.10, 0.20), 0.80 * pct, 0.09,
        boxstyle='round,pad=0.01', facecolor=col, edgecolor='none', alpha=0.85))

    # Percentage ticks
    for pv in [0.25, 0.5, 0.75, 1.0]:
        ax_g.axvline(0.10 + 0.80 * pv, ymin=0.19, ymax=0.32,
                     color=C['border'], lw=0.8)

    ax_g.text(0.5, 0.76, f'{cur_rul:.0f}', ha='center', va='center',
              color=col, fontsize=32, fontweight='bold')
    ax_g.text(0.5, 0.60, 'CYCLES REMAINING', ha='center', va='center',
              color=C['muted'], fontsize=7)
    ax_g.text(0.5, 0.48, f'{pct*100:.0f} % LIFE LEFT', ha='center', va='center',
              color=C['muted'], fontsize=7)
    ax_g.text(0.5, 0.08, ac.upper(), ha='center', va='center',
              color=col, fontsize=8, fontweight='bold',
              bbox=dict(boxstyle='round,pad=0.3', facecolor=C['panel'],
                        edgecolor=col, alpha=0.8))
    ax_g.set_title('RUL GAUGE', color=C['teal'], fontsize=9, fontweight='bold')

    # ── Panel 4: Model agreement heatmap strip ────────────────────────────────
    ax_h.cla(); ax_h.set_facecolor(C['panel'])
    for sp in ax_h.spines.values(): sp.set_edgecolor(C['border'])

    strip = np.vstack([iso_win, ae_win, lstm_win]).astype(float)  # (3, trail)
    custom_cmap = mcolors.LinearSegmentedColormap.from_list(
        'agreement', [C['panel'], C['teal'], C['crimson']], N=128)
    ax_h.imshow(strip, aspect='auto', interpolation='none',
                cmap=custom_cmap, vmin=0, vmax=1,
                extent=[cyc_win[0], cyc_win[-1], -0.5, 2.5])
    ax_h.set_yticks([0, 1, 2])
    ax_h.set_yticklabels(['ISO', 'AE', 'LSTM'], fontsize=8, color=C['text'])
    ax_h.set_xlabel('Cycle', fontsize=8)
    ax_h.set_title('MODEL AGREEMENT STRIP  (teal=anomaly flagged, red=all agree)',
                   color=C['teal'], fontsize=8, fontweight='bold')
    ax_h.grid(False)

    # ── Panel 5: Alert log ────────────────────────────────────────────────────
    if any_anom[-1] and cur_risk > 0.40:
        alerts.append((cur_cyc, st, ac, col))

    ax_a.cla(); ax_a.axis('off'); ax_a.set_facecolor(C['panel'])
    for sp in ax_a.spines.values(): sp.set_edgecolor(C['border'])
    ax_a.set_title('ALERT LOG', color=C['teal'], fontsize=9,
                   fontweight='bold', loc='left', pad=4)

    if alerts:
        for k, (cyc, s, a, c) in enumerate(reversed(alerts[-7:])):
            alpha_val = max(0.25, 1.0 - k * 0.12)
            ax_a.text(0.04, 0.88 - k * 0.13,
                      f'T+{cyc:03d}  ENG-{WATCH_UNIT:02d}  [{s}]',
                      transform=ax_a.transAxes, color=c,
                      fontsize=8, fontweight='bold', alpha=alpha_val)
            ax_a.text(0.04, 0.82 - k * 0.13, f'         → {a}',
                      transform=ax_a.transAxes, color=C['muted'],
                      fontsize=7, alpha=alpha_val * 0.8)
    else:
        ax_a.text(0.5, 0.5, '✓ ALL NOMINAL', transform=ax_a.transAxes,
                  ha='center', va='center', color=C['green'],
                  fontsize=10, fontweight='bold')


ani = FuncAnimation(fig, animate,
                    frames=min(total_cyc - 1, 200),
                    interval=110, repeat=False, blit=False)

ani.save('realtime_monitor.gif', writer='pillow', fps=9, dpi=95)
plt.close()

display(Image('realtime_monitor.gif'))

print('\n  ✓ All outputs saved:')
print('    plot_model_evaluation.png')
print('     plot_fleet.png')
print('     realtime_monitor.gif')

from IPython.display import FileLink
print('\n   Downloads:')
for f in ['realtime_monitor.gif', 'plot_model_evaluation.png', 'plot_fleet.png']:
    display(FileLink(f))